# 02 — Pré-processamento e split (T05–T08)

Não treinar nesta etapa. Não olhar métrica de teste (T12).  
Docs: [limpeza](../docs/limpeza.md) · [correlação](../docs/correlacao.md) · [split](../docs/split.md).

In [ ]:
from pathlib import Path
import sys

import joblib

ROOT = Path.cwd() if (Path.cwd() / "data" / "data.csv").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.preprocess import load_and_clean, load_raw, build_preprocess_pipeline, dump_unfitted
from src.correlacao import save_heatmap, pearson_pairs
from src.train import make_split, assert_disjoint, save_split

bruto = load_raw()
print("shape bruto", bruto.shape)
print("NA por coluna (top)")
print(bruto.isna().sum().sort_values(ascending=False).head(5))
print("duplicatas", int(bruto.duplicated().sum()))

df, X, y = load_and_clean()
print("limpeza", df.attrs["limpeza"])
print("y dtype", y.dtype, "valores", sorted(y.unique()))

## T06 — Pipeline (factory, sem fit no dataset inteiro)

In [ ]:
ct = build_preprocess_pipeline(X)
print(ct)
path = dump_unfitted(ROOT / "models" / "preprocess_unfitted.joblib")
loaded = joblib.load(path)
print("serializado", path.name, type(loaded).__name__)
# proposital: nenhum ct.fit(X) aqui

## T07 — Correlação (Pearson; decisão em docs/correlacao.md)

In [ ]:
fig = save_heatmap(X, ROOT / "reports" / "figures" / "02_correlacao.png")
print(fig.relative_to(ROOT))
print(pearson_pairs(X).head(10))

![Heatmap Pearson](../reports/figures/02_correlacao.png)

Pares |r|>0.90 (raio/perímetro/área) são o mesmo núcleo em unidades diferentes — **não** causalidade clínica. Baseline **mantém as 30** features; ver `docs/correlacao.md`.

## T08 — Split 70/15/15 estratificado, seed 42

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = make_split(X, y)
assert_disjoint(X_train, X_val, X_test)
print("n", len(X), "train", len(X_train), "val", len(X_val), "test", len(X_test))
print("prop maligno", float(y.mean()), float(y_train.mean()), float(y_val.mean()), float(y_test.mean()))
save_split(X_train, X_val, X_test, path=ROOT / "models" / "split.joblib")
print("teste lacrado até T12 — sem métrica em y_test")